<div dir="rtl">

# 02 — סטטיסטיקות ליגה

**פעימה A: הקבוצה בעונה ובין עונות.** השאלה: מה מבדיל אלופה מיורדת, כמה עונה אחת קשורה לבאה וכמה הכסף מסביר?

זו מחברת תיאורית: הנתונים כוללים את סוף העונה. אין כאן פיצ'רים, מודלים, כיוונון או כתיבה לדאטה הנקי. סיכום של עונה אינו קלט למודל באותה עונה; בעתיד מותר להעביר מידע מעונה קודמת דרך `prematch` בלבד.

כל החישובים מוגבלים ל-development: עונות 8–25 (2006/07–2023/24). עונות ההחזקה והעונה החיה אינן נטענות או מוצגות.

## תוכן עניינים — חמש פעימות

1. **A — הקבוצה בעונה ובין עונות** — בפרק הנוכחי.
2. B — הליגה לאורך השנים — יתווסף לאחר אישור A ומיזוג.
3. C — מהלך העונה והטבלה — טרם מומש.
4. D — המשחק והשערים — טרם מומש.
5. E — שופטים, מאמנים, אצטדיונים ומשחקים גדולים — טרם מומש.

## איך קוראים נכון?

נקודות למשחק הן 3 לניצחון ו-1 לתיקו מהמשחקים ששוחקו, ללא הורדות וללא חציית נקודות בעונות הקיזוז. **המיקום הסופי הוא רשמי**: הוא עשוי להיות מושפע מהורדות ומהמשחקים הטכניים שהוחרגו; אינו דירוג ספורטיבי שחושב מחדש.

**השוואות מרובות:** הבדלים בין קבוצות עשויים להופיע במקרה. מציגים גודל מדגם ורווחי 95% במקום דירוג "הכי טוב". Wilson משמש לשיעורי משחקים; bootstrap של עונות משמש לממוצעים ולקשרים. הרווחים תיאוריים, אינם תיקון להשוואות מרובות ואינם מבטיחים עצמאות בין קבוצות ועונות. אין מסקנה מקבוצה עם פחות מ-30 תצפיות; גם 18 עונות אינן 252 ניסויים עצמאיים.

**קשר אינו סיבה:** שווי סוף עונה, משמעת או שינוי מאמן יכולים להיות קשורים לביצועים בלי לגרום להם. רגרסיית השווי כאן היא תיאור בתוך המדגם, ולא בדיקת ניבוי על עתיד.

</div>

<div dir="rtl">

### סביבת התצוגה
השאלה: באילו עונות וקבוצה נתמקד בדוגמאות? מגדירים את 2023/24 ואת מכבי חיפה, ואת אותם צבעי בית/תיקו/חוץ של מחברת 01. כל חישוב מיובא מהחבילה.

</div>

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

from ligat.analysis.league_stats import (
    persistence_summary,
    position_profile,
    promoted_value_check,
    promoted_value_summary,
    season_persistence,
    team_consistency,
    team_season_stats,
    team_style_profile,
    value_summary,
    value_vs_performance,
)
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.viz.league_stats import AWAY_COLOR, DRAW_COLOR, HOME_COLOR, chapter_a_chart

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
EXAMPLE_SEASON = 25
TEAM = "מכבי חיפה"
OUTCOME_COLORS = {"בית": HOME_COLOR, "תיקו": DRAW_COLOR, "חוץ": AWAY_COLOR}
pd.set_option("display.max_columns", 15)
pio.renderers.default = "plotly_mimetype+notebook"
display(
    pd.DataFrame(
        {
            "הגדרה": ["עונת דוגמה", "קבוצה", "עונות מחקר"],
            "ערך": ["2023/24", TEAM, "2006/07–2023/24 בלבד"],
        }
    )
)

<div dir="rtl">

### טעינה מוגבלת לפיתוח
השאלה: כמה משחקים ועונות באמת נכללים? קוראים דרך Repository רק את עונות הפיתוח. הטבלאות הרשמיות שנקראות בתשתית הקיימת מספקות תוויות מיקום וירידה תיאוריות בלבד; אין כאן קלט למודל או אימון. [משרת: הבנת הליגה]

</div>

In [ ]:
tables = load_analysis_tables(Repository(), seasons=DEVELOPMENT_SEASONS)
display(
    pd.DataFrame(
        {
            "תצוגה": ["משחקים", "קבוצה × משחק", "קבוצה × עונה"],
            "שורות": [len(tables.match_table), len(tables.team_matches), len(tables.team_seasons)],
        }
    )
)

<div dir="rtl">

## A — הקבוצה בעונה ובין עונות
### עונת דוגמה: 2023/24
השאלה: איך נראית עונה של קבוצה במספרים? מציגים 15 עמודות נבחרות. אין קיזוז או עונשים בנקודות; המיקום הסופי רשמי. שערים ותוצאות משתמשים גם במשחקי ht_conflict — החרגות המחצית ותזמון השערים שייכות לפעימות הבאות. [משרת: הבנת הליגה, goals]

</div>

In [ ]:
season_stats = team_season_stats(tables)
display(
    season_stats.xs(EXAMPLE_SEASON, level="season_id")
    .sort_values("final_position")[
        [
            "team_name",
            "final_position",
            "matches_played",
            "wins",
            "draws",
            "losses",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "home_points_per_match",
            "away_points_per_match",
            "big_win_rate",
            "btts_rate",
            "over_2_5_rate",
            "relegated",
        ]
    ]
    .round(3)
)

<div dir="rtl">

### הקבוצה שבחרנו לאורך עונות הפיתוח
השאלה: האם המאפיינים של מכבי חיפה קבועים? משנים רק את TEAM בתא ההגדרות. מוצגות שמונה העונות האחרונות הזמינות, ללא שורות החזקה. [משרת: הבנת הליגה]

</div>

In [ ]:
display(
    season_stats.loc[
        season_stats.team_name.eq(TEAM),
        [
            "team_name",
            "final_position",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "longest_unbeaten_run",
            "longest_winless_run",
        ],
    ]
    .tail(8)
    .round(3)
)

<div dir="rtl">

### כמה שווה כל מקום — מפרידים עידנים
השאלה: מהו טווח הנקודות למשחק בכל מיקום? לכל עידן ומיקום מוצגים n, ממוצע, חציון וקצוות. רווח הממוצע הוא bootstrap של עונות. אין להסיק ממיקום בעל מעט עונות; מיקום 6 בליגה של 12 אינו אותו הקשר כמו בליגה של 14. [משרת: motivation, table_model]

</div>

In [ ]:
profiles = position_profile(tables)
display(
    profiles.xs("14_teams", level="era")[
        [
            "n",
            "points_per_match_mean",
            "points_per_match_median",
            "points_per_match_min",
            "points_per_match_max",
            "points_per_match_ci_lower",
            "points_per_match_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 1 — התפלגות נקודות לפי מיקום
השאלה: כמה רחבה התפלגות הנקודות בכל מקום בעידן 14 הקבוצות? כל נקודה היא עונת קבוצה; הקופסה מציגה את הפיזור בפועל ולא תחזית לעונה הבאה. n מופיע בריחוף, וטבלת הרווחים נמצאת מעל. [משרת: motivation, table_model]

</div>

In [ ]:
chapter_a_chart("position", season_stats).show()

<div dir="rtl">

### גרף 2 — התקפה והגנה יחד
השאלה: מה מבדיל בין עונה של קבוצה בצמרת לעונה בתחתית? השערים נלקחים מהמשחקים, והצבע מייצג מיקום רשמי. כל נקודה מתארת עונה שלמה, לא מדידת כוח לפני משחק. מספר המשחקים מופיע בריחוף. [משרת: goals]

</div>

In [ ]:
chapter_a_chart("attack_defense", season_stats).show()

<div dir="rtl">

### גרף 3 — בית מול חוץ
השאלה: מי צברה יותר נקודות בבית לעומת בחוץ? מתחת לאלכסון יש יותר נקודות בבית. הממוצע בכותרת נותן משקל שווה לכל עונת קבוצה, אינו אומדן סיבתי של יתרון הביתיות. [משרת: goals, prematch]

</div>

In [ ]:
chapter_a_chart("home_away", season_stats).show()

<div dir="rtl">

### גרף 4 — ניצחונות גדולים מול צרים
השאלה: האם שיעור ניצחונות גבוה מורכב מהכרעות בהפרש גדול או קטן? השיעורים מחולקים בכל המשחקים, לא רק בניצחונות: ניצחון גדול הוא הפרש 3 ומעלה וצָר הוא הפרש 1. n ורווחי Wilson של 95% נמצאים בריחוף. [משרת: goals, scores]

</div>

In [ ]:
chapter_a_chart("win_style", season_stats).show()

<div dir="rtl">

### סגנון משחק על פני לפחות חמש עונות
השאלה: אצל אילו קבוצות שתי הקבוצות מבקיעות, ואצל אילו יש לפחות שלושה שערים? אלה ממוצעים של שיעורי העונות, במשקל שווה לעונה. רווחי 95% חושבו בדגימת עונות, ומוצגים גם מספר העונות והמשחקים. לא נדרג איכות או נסיק סיבה. [משרת: scores, goals]

</div>

In [ ]:
styles = team_style_profile(tables)
display(
    styles[
        [
            "team_name",
            "n_seasons",
            "n_matches",
            "btts_rate",
            "btts_rate_ci_lower",
            "btts_rate_ci_upper",
            "over_2_5_rate",
            "over_2_5_rate_ci_lower",
            "over_2_5_rate_ci_upper",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 5 — קבוצות פתוחות וסגורות
השאלה: כמה רחב טווח השיעורים בין הקבוצות? הקווים מראים רווחי 95% עבור ממוצעי העונות, והריחוף מציג את גודל המדגם. אין כאן טענה שקבוצה מסוימת תמיד תשחק כך. [משרת: scores]

</div>

In [ ]:
chapter_a_chart("open_style", styles).show()

<div dir="rtl">

### רצף בין עונות — מי באמת שיחקה גם ב-t−1?
השאלה: האם העונה הקודמת קשורה לבאה? מחברים רק אותה קבוצה בשתי עונות עוקבות שנבחרו; לא מדלגים מעל עונת היעדרות. עונה 8 ועולות ללא עונה קודמת בליגה אינן בזוגות. [משרת: prematch, table_model]

</div>

In [ ]:
pairs = season_persistence(tables)
display(pairs.head(8).round(3))

<div dir="rtl">

### מתאם ומקדם התמדה
השאלה: כמה מהפער בעונה הקודמת נשאר בקשר הליניארי? מציגים מתאם פירסון, ספירמן ושיפוע לצד n ורווחי 95% בדגימת בלוקים של עונות. השיפוע אינו אחוז דיוק: אפילו שיפוע 1 אינו מוכיח חיזוי מושלם. מתאם המיקומים מערב פורמטים ומיקומים רשמיים, ולכן יש לקרוא אותו בזהירות. [משרת: prematch]

</div>

In [ ]:
persistence = persistence_summary(tables)
display(
    persistence[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "slope",
            "intercept",
            "slope_ci_lower",
            "slope_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 6 — העונה הקודמת מול הנוכחית
השאלה: האם קו ההתמדה קרוב לקו 45 מעלות? הנתונים הם זוגות עונות, והקו הוא רגרסיה תיאורית על הפיתוח. לא נשתמש בו כתוצאת חיזוי מחוץ לאימון. [משרת: prematch]

</div>

In [ ]:
chapter_a_chart("persistence", pairs, persistence).show()

<div dir="rtl">

### כסף וביצועים — שני מושגים שונים של שווי
השאלה: האם שווי קודם קשור לנקודות, והאם הקשר שונה משווי באותה עונה? שווי t−1 הוא נתח בליגה הקודמת. שווי אותה עונה הוא סגל הכולל את מי שהיה במהלך העונה, ולכן תיאורי בלבד. חוסרי שווי קודם: העונה הראשונה, עולות וערך חסר במקור; אין השלמה כאן. [משרת: goals]

</div>

In [ ]:
values = value_vs_performance(tables)
display(values.xs(EXAMPLE_SEASON, level="season_id").head(8).round(3))

<div dir="rtl">

### סיכום קשרי השווי
השאלה: כמה זוגות מלאים תומכים בכל מתאם? ספירמן משווה סדר, ופירסון קשר ליניארי. מוצג גם גודל הסגל; הוא מספר השחקנים בדף העונה, לא בהכרח הסגל ביום משחק. הרווחים מבוססים על דגימת עונות. [משרת: goals]

</div>

In [ ]:
value_relationships = value_summary(tables)
display(
    value_relationships[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "spearman_ci_lower",
            "spearman_ci_upper",
            "slope",
            "intercept",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 7 — חריגות לעומת הקשר הממוצע לשווי קודם
השאלה: מי צברה יותר או פחות נקודות מהקשר הליניארי בתוך המדגם? עד עשר חריגות חיוביות ועשר שליליות מסומנות בשם ובעונה. over_under הוא שארית תיאורית, אינו שגיאת תחזית חיצונית ואינו הוכחה לאיכות ניהול. קבוצות ללא שווי קודם נשארות בטבלה עם NA ואינן מסומנות ברגרסיה. [משרת: goals]

</div>

In [ ]:
chapter_a_chart("value", values, value_relationships).show()

<div dir="rtl">

### עולות וכלל ההשלמה — בדיקה בדיעבד
השאלה: איך שווי סגל העולה באותה עונה משתווה לחציון החצי התחתון לפי שווי? החצי התחתון נבחר מערכי אותה עונה הזמינים, ובמספר אי-זוגי נלקח floor(n/2). זה אינו פיצ'ר לפני משחק: שני הערכים משתמשים בסגל העונתי בדיעבד. גם העולה עשויה להשתתף בקבוצת הייחוס. [משרת: prematch]

</div>

In [ ]:
promoted = promoted_value_check(tables)
display(promoted.head(10).round(4))

<div dir="rtl">

### סיכום בדיקת ההשלמה
השאלה: בכמה עולות הערך הממשי נמוך מהערך לפי הכלל? fraction_below מחושב רק מזוגות מלאים; n_missing מדווח בנפרד. אין שינוי בכלל ההשלמה שאושר ואין מילוי של הערכים החסרים בדאטה. [משרת: prematch]

</div>

In [ ]:
promoted_summary = promoted_value_summary(tables)
display(promoted_summary.round(4))

<div dir="rtl">

### גרף 8 — שווי ממשי מול ערך הכלל
השאלה: כמה רחוקים הערכים מהאלכסון? זו בדיקת סבירות תיאורית של הכלל, לא אימות של שווי ביום פתיחת העונה ולא החלטה חדשה. חוסר נשאר חסר. [משרת: prematch]

</div>

In [ ]:
chapter_a_chart("promoted", promoted).show()

<div dir="rtl">

### יציבות ויו-יו בין עונות
השאלה: מי עקבית בנקודות למשחק, ומי עלתה לפחות פעמיים בתקופת הפיתוח? הממוצע נותן משקל שווה לעונה; סטיית התקן חסרה אם יש פחות משלוש עונות. עליות בעונה הראשונה אינן ידועות. מיקומי שיא ושפל רשמיים, ומספרי עליות וירידות נצפו רק בפיתוח. [משרת: table_model, הבנת הליגה]

</div>

In [ ]:
consistency = team_consistency(tables)
display(
    consistency[
        [
            "team_name",
            "seasons",
            "mean_ppm",
            "std_ppm",
            "mean_ppm_ci_lower",
            "mean_ppm_ci_upper",
            "best_final_position",
            "worst_final_position",
            "titles",
            "top3_seasons",
            "relegations",
            "promotions",
            "is_yoyo",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 9 — יציבות מול ממוצע
השאלה: האם ממוצע גבוה מגיע עם תנודתיות קטנה? גודל נקודה הוא מספר העונות, ובריחוף נראים רווחי הממוצע והיסטוריית העליות. קבוצות עם פחות משלוש עונות אינן בגרף, ואין פירוש סיבתי לצבע היו-יו. [משרת: table_model]

</div>

In [ ]:
chapter_a_chart("consistency", consistency).show()

<div dir="rtl">

## מה למדנו — פעימה A

1. הקשר בין נקודות למשחק בעונות עוקבות חיובי: מקדם ההתמדה הוא כ-0.77, ולא "חצי מהפער"; זה מצדיק מחקר של t−1 ב-**prematch**, בלי לטעון לדיוק חיזוי של 77%.
2. נתח שווי בעונה הקודמת קשור לנקודות למשחק עם ספירמן כ-0.56 על 202 זוגות מלאים; התרומה לחיזוי של **goals** עדיין צריכה להיבדק מחוץ לאימון.
3. שווי באותה עונה מציג קשר חזק יותר, אך הוא כולל מידע מסוף העונה ולכן נשאר תיאורי ולא נכנס ל-**goals** של אותה עונה.
4. ב-25 מתוך 36 עולות עם שווי זמין, כ-69%, השווי העונתי נמוך מהערך לפי כלל ההשלמה; עולה אחת חסרה במקור, וזו בדיקת סבירות ל-**prematch** בלבד — הכלל לא שונה.
5. הפרדת עידנים, הצגת מספר עונות ורווחי אי-ודאות, והבחנה בין מיקום רשמי לנקודות על המגרש דרושות להמשך מחקר **motivation** ו-**table_model**.
6. שיעורי משחקים פתוחים וגדולים מתארים את הפיתוח ומספקים שאלות ל-**scores** ו-**goals**; הם אינם פיצ'רים של אותה עונה ואינם ראיה לסיבתיות.

פעימות B–E טרם מומשו. נעצור ל-QA ולאישור מיזוג A לפני הפרק הבא.

</div>